# 🔁 GIADA Task 7 — Ca_HVA in circuito chiuso, un compartimento
Microcanary diagnostico: NEURON autentico contro formula, LUT-513 e physical-τ congelati. Ogni braccio produce il proprio voltaggio. Non è ancora il neurone Hay completo.

In [ ]:
from pathlib import Path
import base64, hashlib, json, os, shutil, subprocess, sys
from IPython.display import Javascript, display
WORK=Path('/kaggle/working/giada_task_7'); GIADA_REPO=WORK/'giada'; TEACHER_REPO=WORK/'neuron_as_deep_net'
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(GIADA_REPO)],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER_REPO)],check=True)
subprocess.run(['git','-C',str(TEACHER_REPO),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(GIADA_REPO),'rev-parse','HEAD'],text=True).strip(); print({'giada_revision':REVISION})


In [ ]:
if shutil.which('nrnivmodl') is None:
    subprocess.run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7'],check=True)
    os.environ['PATH']=str(Path(sys.executable).parent)+os.pathsep+os.environ.get('PATH','')
assert shutil.which('nrnivmodl'),'nrnivmodl non trovato dopo installazione NEURON'
assert shutil.which('gcc'),'gcc non disponibile'
import neuron, torch
assert torch.cuda.is_available(),'La Task 7 richiede GPU CUDA per il confronto con il modello congelato.'
print({'neuron':neuron.__version__,'cuda':torch.cuda.get_device_name(0),'nrnivmodl':shutil.which('nrnivmodl')})


In [ ]:
sys.path.insert(0,str(GIADA_REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]: del sys.modules[name]
from src.giada_teacher import ExtractedGateFormula,compile_nmodl,ClosedLoopCaHVAConfig,run_closed_loop_microcanary
from src.giada_teacher.voltage_path_stress import EXPECTED_TASK5_ARCHIVE_SHA256,EXPECTED_TASK5_REPORT_SHA256,verified_task5_root
prereg=json.loads((GIADA_REPO/'experiments/teacher_cahva_closed_loop_microcanary_preregistration_v1.json').read_text())
display({'scope':prereg['scope'],'episode_count':prereg['model']['episode_count'],'arms':prereg['arms']})


In [ ]:
def file_sha(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda:handle.read(1024*1024),b''): digest.update(chunk)
    return digest.hexdigest()
INPUT_ROOT=Path('/kaggle/input'); override=os.environ.get('GIADA_TASK5_ARTIFACT')
candidates=[Path(override).expanduser()] if override else []
if INPUT_ROOT.is_dir():
    candidates += list(INPUT_ROOT.rglob('giada_primitive_scaling_laws.zip'))
    candidates += list(INPUT_ROOT.rglob('archive.zip'))
    candidates += [p.parent for p in INPUT_ROOT.rglob('final_report.json') if (p.parent/'frozen_scaling_checkpoints.pt').is_file()]
def exact(path):
    try: return file_sha(path)==EXPECTED_TASK5_ARCHIVE_SHA256 if path.is_file() else file_sha(path/'final_report.json')==EXPECTED_TASK5_REPORT_SHA256
    except Exception: return False
TASK5_SOURCE=next((p.resolve() for p in candidates if p.exists() and exact(p)),None)
assert TASK5_SOURCE is not None,'Aggiungi agli Input Kaggle giada_primitive_scaling_laws.zip esatto oppure imposta GIADA_TASK5_ARTIFACT.'
TASK5_ROOT=verified_task5_root(TASK5_SOURCE,Path('/kaggle/working/.task7_task5_verified'))
print({'task5_source':str(TASK5_SOURCE),'exact_hash_verified':True})


In [ ]:
MOD=TEACHER_REPO/'L5PC_NEURON_simulation/mods/Ca_HVA.mod'
formula=ExtractedGateFormula.from_mod(MOD)
mechanism_root=compile_nmodl(MOD,WORK/'compiled_mechanism')
OUTPUT_DIR=Path('/kaggle/working/artifacts/giada_cahva_closed_loop_microcanary')
assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
config=ClosedLoopCaHVAConfig()
print({'mechanism_compiled':str(mechanism_root),'output':str(OUTPUT_DIR),'steps_per_episode':int(config.duration_ms/config.dt_ms)})


## 🧪 Circuito chiuso causale
La formula esatta calibra prima il solver di membrana. Il report viene salvato anche se quel gate fallisce, così l'artefatto forense non va perso. Una riga per episodio mostra avanzamento ed ETA.

In [ ]:
report=run_closed_loop_microcanary(formula,TASK5_ROOT,mechanism_root,OUTPUT_DIR,config,code_revision=REVISION)
compact={key:{'formula_V':round(row['formula']['voltage_rmse_mv'],5),'lut_V':round(row['lut']['voltage_rmse_mv'],5),'physical_V':{seed:round(v['voltage_rmse_mv'],5) for seed,v in row['physical'].items()}} for key,row in report['episodes'].items()}
display({'valid':report['valid'],'reference_solver_calibrated':report['reference_solver_calibrated'],'formula_worst_voltage_rmse_mv':report['formula_worst_voltage_rmse_mv'],'formula_worst_gate_rmse':report['formula_worst_gate_rmse'],'full_642_segment_teacher_tested':report['full_642_segment_teacher_tested'],'trained_voltage_network_tested':report['trained_voltage_network_tested'],'episodes':compact})
if not report['reference_solver_calibrated']: print('ATTENZIONE: mismatch del solver di base; non attribuire gli errori a LUT o rete. Scarica comunque lo ZIP.')


## 📦 Scarica lo ZIP con il metodo Blob/base64

In [ ]:
archive=Path(shutil.make_archive('/kaggle/working/giada_cahva_closed_loop_microcanary','zip',OUTPUT_DIR.parent,OUTPUT_DIR.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
